# Amsterdam: a case built in Python

`run_file` ([the first tutorial](delft.ipynb)) does three things before it runs: it builds a network, a timetable and
the travellers from the case's files. This tutorial does the same steps one by one in Python on a larger case,
Amsterdam's morning (198 252 travellers), the way you would with tables of your own.

It needs `pip install openmobisim pyarrow`. Everything below was run as shown, on a laptop (Apple M2 Pro, 12 cores).
The travellers are synthetic and the choice coefficients are the defaults, not fitted to Amsterdam: the results show
the tool at work and are not a description of the city.

In [1]:
import urllib.request, zipfile

urllib.request.urlretrieve("https://zenodo.org/records/24585967/files/amsterdam.zip", "amsterdam.zip")
zipfile.ZipFile("amsterdam.zip").extractall()

## The settings

The case's `run.toml` holds the window, the classes of travellers and every parameter. It is plain TOML.

In [2]:
import tomllib

with open("amsterdam/run.toml", "rb") as f:
    settings = tomllib.load(f)
window = (settings["window_start_s"], settings["window_end_s"])   # seconds after midnight: 07:30 to 09:30
classes = settings["demand"]["classes"]
print(window, classes)
print({name: value for name, value in settings["parameters"].items() if name != "coefficients"})

(27000, 34200) ['bike_only', 'car_and_bike', 'car_only', 'from_outside_car', 'from_outside_transit', 'goods', 'no_vehicle']
{'capacity_per_lane': 1800.0, 'bike_speed': 15.0, 'walk_speed': 4.8, 'stop_walk_max': 1500.0, 'transfer_walk_max': 400.0, 'boarding_slack': 60, 'max_rides': 4, 'nearby_stops': 3, 'max_car_routes': 6, 'max_transit_journeys': 10, 'choice': 'logit', 'nest_scale': 0.5, 'step_exponent': 0.75, 'iteration_cap': 50, 'iteration_floor': 0, 'link_bin': 300, 'seed': 0}


Every parameter in the file is at its default ([the parameters table](../reference/parameters.md)), and so are the choice
coefficients of every class, so the calls below give none of them: a keyword left out takes its default.

## The network, the timetable, the travellers

A table is given in memory (a dict of lists, a pandas or polars frame, a pyarrow table) or, as here, as the path of a
Parquet file. The timetable is read on the network, and the travellers on both: each trip end is linked to the
network per mode and to the stops within walking distance.

In [3]:
import time
import openmobisim as oms

start = time.perf_counter()
network = oms.Network.from_tables("amsterdam/nodes.parquet", "amsterdam/links.parquet")
transit = oms.Transit.from_gtfs("amsterdam/gtfs.zip", network)
demand = oms.Demand.from_table("amsterdam/trips.parquet", classes, network, transit=transit)
print(f"built in {time.perf_counter() - start:.1f} s")
print("repairs:", network.repairs, transit.repairs, demand.repairs)

built in 3.3 s
repairs: {} {} {}


A reader sets right or leaves out what it cannot read instead of refusing the whole table (a link naming a node the
nodes table does not hold is left out, a trip of a class not given is dropped), says so in a warning and counts it in
`repairs`; these files need none.

## The run

In [4]:
start = time.perf_counter()
run = oms.run(network, demand, transit, window=window, out="amsterdam-run")
print(f"run time {time.perf_counter() - start:.1f} s; ended by the stopping rule: {run.stopped_by_rule}")

run time 51.5 s; ended by the stopping rule: True


In [5]:
import pyarrow.parquet as pq

def by_mode(trips_file):
    """Per mode: the travellers who took it, their share (%) and their mean door-to-door time (min)."""
    trips = pq.read_table(trips_file).to_pylist()
    rows = {}
    for mode in ("car", "bike", "walk", "transit"):
        times = [t["travel_time_s"] for t in trips if t["mode"] == mode and t["finished"]]
        count = sum(t["mode"] == mode for t in trips)
        rows[mode] = (count, 100 * count / len(trips), sum(times) / len(times) / 60)
    return rows

print("mode     travellers  share (%)  mean door-to-door time (min)")
for mode, (count, share, minutes) in by_mode(run.trips).items():
    print(f"{mode:8} {count:10} {share:10.1f} {minutes:10.1f}")
print("unfinished at 09:30:", pq.read_table(run.trips, columns=["finished"])["finished"].to_pylist().count(False))

mode     travellers  share (%)  mean door-to-door time (min)


car           93729       47.3       11.7
bike          41725       21.0       22.7
walk           6254        3.2       26.7
transit       56544       28.5       36.1
unfinished at 09:30: 0


These are the results `oms.run_file("amsterdam", out=...)` gives: the same inputs and settings give the same trips,
links and routes tables.

## Who takes what, by class

A traveller class has its own choice coefficients (here all classes have the default ones), and each traveller has
the modes open to it (the column `modes` of the trips table); in these cases the modes follow the class. The shares
by class show what the modes open to a traveller do to the choice.

In [6]:
import collections

trips = pq.read_table(run.trips, columns=["class_id", "mode"]).to_pylist()
counts = collections.Counter((t["class_id"], t["mode"]) for t in trips)
print("class                 travellers  car (%)  bike (%)  walk (%)  transit (%)")
for class_id in classes:
    total = sum(counts[class_id, mode] for mode in ("car", "bike", "walk", "transit"))
    shares = "".join(f"{100 * counts[class_id, mode] / total:9.1f}" for mode in ("car", "bike", "walk", "transit"))
    print(f"{class_id:21} {total:10} {shares}")

class                 travellers  car (%)  bike (%)  walk (%)  transit (%)
bike_only                  30926       0.0     90.9      1.3      7.7
car_and_bike               59002      74.6     22.6      0.7      2.1
car_only                   14721      95.9      0.0      1.1      3.0
from_outside_car           28856     100.0      0.0      0.0      0.0
from_outside_transit       14988       0.0      1.8      0.0     98.2
goods                       6750     100.0      0.0      0.0      0.0
no_vehicle                 43009       0.0      0.0     12.3     87.7


The classes `from_outside_car` and `goods` have the car alone; `from_outside_transit` has transit, and a few of its
travellers also a bike. Among the others, a traveller with a car and a bike drives three times in four, one with a bike and no car cycles nine times
in ten, and one with neither takes transit seven times in eight.

## Distance travelled on the network

`links.parquet` counts the cars, bikes and walkers that entered each link in each time bin. With the links' lengths
it gives the distance travelled on the network per mode. The walkers counted on links are those of walk trips; the
walks to a stop or between a door and the network are not on links.

In [7]:
network_links = pq.read_table("amsterdam/links.parquet", columns=["link_id", "length_m"])
length_m = dict(zip(network_links["link_id"].to_pylist(), network_links["length_m"].to_pylist()))
for mode, column in (("car", "cars"), ("bike", "bikes"), ("walk", "walkers")):
    table = pq.read_table(run.links, columns=["link_id", column])
    km = sum(n * length_m[l] for l, n in zip(table["link_id"].to_pylist(), table[column].to_pylist())) / 1000
    print(f"{mode:5} {km:9.0f} km on the network, {km / by_mode(run.trips)[mode][0]:5.2f} km per traveller of that mode")

car      991629 km on the network, 10.58 km per traveller of that mode


bike     222087 km on the network,  5.32 km per traveller of that mode


walk      12822 km on the network,  2.05 km per traveller of that mode


## The pages

The map and the summary are written from the finished run. A title replaces the file's name at the top of the page.

In [8]:
import os

run.map("amsterdam-map.html", title="Amsterdam, 07:30 to 09:30")
run.summary("amsterdam-summary.html", title="Amsterdam, 07:30 to 09:30")
for page in ("amsterdam-map.html", "amsterdam-summary.html"):
    print(f"{page}: {os.path.getsize(page) / 1e6:.1f} MB")

amsterdam-map.html: 16.8 MB
amsterdam-summary.html: 12.4 MB


![The map page of the run: Amsterdam's roads, one ribbon per direction, coloured by volume over capacity](../_static/figures/amsterdam_map.png)

The map page as it opens (the network is from OpenStreetMap, © OpenStreetMap contributors).

## The record

`run_record.toml` holds the settings the run ran with, the defaults included, and what the run found. Its first part,
copied into a `run.toml`, runs the same run again; the fingerprints of the inputs in its second part tell whether an
input file has changed since.

In [9]:
record = open(run.record).read()
print(record[record.index("[result]\n"):record.index("[result.transit.gtfs_report]")].strip())

[result]
openmobisim_version = "0.2.0-dev"
window_end_used_s = 34200  # s: window_end_s, moved up to a whole link_bin when it was not one
iterations = 9
stopped_by_rule = true

[result.network]
source = "tables"
nodes = 121502
links = 489167
fingerprint = "7d1318422a05ef4c"

[result.demand]
trips = 198252
fingerprint = "c25b5ba10e7574ad"

[result.transit]
stops = 4003
runs = 7703
fingerprint = "f6f58a602c2f86ee"


## Where to go from here

- [Guides](../guides/index.md): one task per page, such as a case from your own data, the settings file, reading the
  results, or the window.
- [Concepts and methods](../concepts/index.md): how the model works, starting with
  [How a run works](../concepts/architecture.md).
- [Reference](../reference/index.md): every function, parameter, input column and output column.